# Pizzeria Amalfi – Liquidity Overview 2025

This notebook analyses the sample bank statement of Pizzeria Amalfi (`memory/reference/sample-data/pizzeria-amalfi-bank-statement-2025-v6.csv`).
It is fully standalone: it reads the CSV directly and does not depend on the application backend.

The statement contains individual transactions only (no balance column), so the daily balance is reconstructed as the cumulative sum of amounts on top of a configurable opening balance.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

CSV_NAME = "memory/reference/sample-data/pizzeria-amalfi-bank-statement-2025-v6.csv"
OPENING_BALANCE = 0.0

csv_path = next(p for p in (Path(CSV_NAME), Path("..") / CSV_NAME) if p.exists())

df = pd.read_csv(csv_path, sep=";", decimal=",", encoding="utf-8")
df["Booking Date"] = pd.to_datetime(df["Booking Date"], format="%d.%m.%y")
df = df.sort_values("Booking Date").reset_index(drop=True)

print(f"Loaded {len(df)} transactions from {csv_path}")
print(f"Period: {df['Booking Date'].min():%Y-%m-%d} – {df['Booking Date'].max():%Y-%m-%d}")
df.head()

## Daily balance

Net cash flow is aggregated per booking day and accumulated into an end-of-day balance. Days without transactions carry the previous balance forward.

In [ ]:
daily_net = df.groupby("Booking Date")["Amount"].sum()
full_range = pd.date_range(daily_net.index.min(), daily_net.index.max(), freq="D")
daily_balance = OPENING_BALANCE + daily_net.reindex(full_range, fill_value=0).cumsum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_balance.index, daily_balance.values, color="#1f77b4", linewidth=1.5)
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_title("Daily account balance (EUR)")
ax.set_ylabel("EUR")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Lowest balance: {daily_balance.min():,.2f} EUR on {daily_balance.idxmin():%Y-%m-%d}")
print(f"Closing balance: {daily_balance.iloc[-1]:,.2f} EUR")

## Monthly inflows vs. outflows

Positive amounts are treated as inflows and negative amounts as outflows. The line shows the net result of each month.

In [ ]:
month = df["Booking Date"].dt.to_period("M")
monthly = pd.DataFrame({
    "Inflows": df["Amount"].clip(lower=0).groupby(month).sum(),
    "Outflows": df["Amount"].clip(upper=0).groupby(month).sum(),
})
monthly["Net"] = monthly["Inflows"] + monthly["Outflows"]

labels = monthly.index.strftime("%b")
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(labels, monthly["Inflows"], color="#2ca02c", label="Inflows")
ax.bar(labels, monthly["Outflows"], color="#d62728", label="Outflows")
ax.plot(labels, monthly["Net"], color="black", marker="o", label="Net")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_title("Monthly inflows and outflows (EUR)")
ax.set_ylabel("EUR")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

monthly.round(2)